In [1]:
print("Yes")

Yes


In [ ]:
import pandas as pd
import numpy as np

def fetch_and_prepare_data():
    # fetching data from the UCL repository
    url= "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
    try:
        df=pd.read_excel(url)
        print(f"{len(df)} transactions.")
    except Exception as e:
        print(f"Error: {e}")
        return None

    df = df.dropna(subset=["StockCode", "Description", 'Quantity'])
    df= df[df["Quantity"]>0]
    df["InvoiceDate"]= pd.to_datetime(df["InvoiceDate"])


    # Transforming into an FMCG Simulation

    top_products=df["StockCode"].value_counts().index
    target_sku=top_products[0] # frequent item 

    product_df= df[df["StockCode"]== target_sku].copy()
    daily_sales=product_df.groupby([product_df["InvoiceDate"].dt.date, "Country"])["Quantity"].sum().reset_index()
    daily_sales["Date"]=pd.to_datetime(daily_sales["Date"])

    # Data sorting
    daily_sales= daily_sales.sort_values(by=["Region","Date"]).reset_index(drop=True)

    # Base assumption is that the region baseline stock capacity of 1000 and try to track sales deplete that the stock over time

    capacity=1000
    reorder_threshold=200

    simulated_records = []

    for region, group in daily_sales.groupby("Region"):
        current_stock=capacity
        for idx, row in group.iterrows():
            units_sold=row["Units_Sold"]
            current_stock -= units_sold

            if current_stock < 0:
                current_stock=0

            reorder_triggered = 1 if current_stock <= reorder_threshold else 0

            simulated_records.append({
                "Date": row["Date"],
                "Region": region,
                "Units_Sold": units_sold,
                "Current_Stock": current_stock,
                "Reorder_Triggered": reorder_triggered
            })

            if reorder_triggered == 1:
                current_stock = capacity

    sandbox_fmcg_df = pd.DataFrame(simulated_records)

    print(f" Engineered {len(sandbox_fmcg_df)} clean history")
    return sandbox_fmcg_df
if __name__ == "__main__":
    fmcg_data= fetch_and_prepare_data()

    if fmcg_data is not None:
        print(f"Processed Supply Chain Features {fmcg_data.head(15)}")

        fmcg_data.to_csv(r"Sandbox_fmcg_pipeline.csv", index=False)
        print("Saved the data ")